# Protein Language Model Embeddings Generator

Turn a FASTA file into a per-protein (or per-residue) embeddings `.h5` that the
[Preparation notebook](https://colab.research.google.com/github/tsenoner/protspace/blob/main/apps/protspace/notebooks/ProtSpace_Preparation.ipynb)
and `protspace prepare` read directly.

1. Pick the model and options in **1. Settings**, and enter your FASTA's path there, or leave it
   empty to upload one from your computer when step 4 asks.
2. **Runtime → Change runtime type → T4 GPU**: on a CPU runtime even the small models are slow.
3. **Runtime → Run all**. The first run in a fresh runtime may stop in **2. Install** and ask you
   to restart the session (*Runtime → Restart session*); run all again after that.

Each step prints what it did, and a failing step says why and what to change.

In [ ]:
# @title 1. Settings
# @markdown Choose a protein language model:
model_name = "Rostlab/prot_t5_xl_half_uniref50-enc"  # @param ["Rostlab/prot_t5_xl_half_uniref50-enc", "Rostlab/ProstT5_fp16", "ElnaggarLab/ankh-base", "ElnaggarLab/ankh-large", "ElnaggarLab/ankh3-large", "facebook/esm2_t6_8M_UR50D", "facebook/esm2_t12_35M_UR50D", "facebook/esm2_t30_150M_UR50D", "facebook/esm2_t33_650M_UR50D", "EvolutionaryScale/esm3-sm-open-v1", "EvolutionaryScale/esmc-300m-2024-12", "EvolutionaryScale/esmc-600m-2024-12"]

# @markdown Choose embedding type: one vector per protein (what ProtSpace plots), or one per residue:
embedding_type = "per_prot"  # @param ["per_prot", "per_res"]

# @markdown Maximum sequence length (longer sequences are skipped and listed):
max_sequence_length = 2000  # @param {type:"integer"}

# @markdown Max batch size for transformer models (halved automatically on GPU out-of-memory):
batch_size = 128  # @param {type:"integer"}

# @markdown FASTA path in this runtime or in Google Drive; leave empty to upload one in step 4:
fasta_filename = ""  # @param {type:"string", placeholder:"/content/drive/MyDrive/proteins.fasta"}

# @markdown Use the UniProt accession found in each FASTA header as the protein ID
# @markdown (`sp|P12345|NAME` becomes `P12345`); unchecked keeps the full header:
should_extract_uniprot_id = False  # @param {type:"boolean"}

# The EvolutionaryScale models run through the `esm` SDK, which step 2 installs only
# for them: it pins transformers below 5 and so replaces Colab's transformers.
needs_esm_sdk = model_name.startswith("EvolutionaryScale/")
print(f"Model: {model_name} ({embedding_type}); sequences up to {max_sequence_length} residues.")
if needs_esm_sdk:
    print("This model needs the EvolutionaryScale `esm` SDK and a Hugging Face login "
          "(step 'Optional: Hugging Face login').")

In [ ]:
# @title 2. Install
# Installed via subprocess, not `!pip`: `%%capture` only works on line 1, which
# `# @title` must occupy for Colab to render the form header, and `-qqq` alone
# does not silence Colab's download bars or pip's dependency-conflict block.
# capture_output hides both; the output is shown only if the install fails.
# --- install helper: identical in every ProtSpace Colab notebook (pinned by a test) ---
import importlib
import importlib.metadata as _md
import subprocess
import sys
import tempfile


def _loaded_distributions():
    """{distribution: installed version} for each one an imported module comes from."""
    importlib.invalidate_caches()
    owners = _md.packages_distributions()
    loaded = set(sys.modules)
    found = {}
    for module in [m for m in loaded if "." not in m]:
        for dist in owners.get(module, ()):
            try:
                # A namespace package (`google`) is shared by many distributions:
                # count one only if a subpackage it ships is loaded.
                if getattr(sys.modules[module], "__file__", None) is None:
                    subs = {
                        f.parts[1] for f in _md.distribution(dist).files or ()
                        if len(f.parts) > 2 and f.parts[0] == module
                    }
                    if not any(f"{module}.{s}" in loaded for s in subs):
                        continue
                found[dist] = _md.version(dist)
            except _md.PackageNotFoundError:
                pass
    return found


def _install(packages):
    """Install `packages` without replacing what this session has already loaded.

    Colab imports numpy, pandas and friends before the first cell runs. pip
    replaces one when a dependency declares a newer version, and then the old
    copy stays in memory and the next import mixes the two (numpy's
    "_blas_supports_fpe" error). protspace itself accepts Colab's versions; its
    biocentral-api client pins tqdm>=4.68 and, through biotrainer-core,
    numpy>=2.4.1, pandas>=2.3.3 and h5py>=3.15, yet uses only biotrainer-core's
    FASTA reader. uv's --override holds every loaded distribution at the
    version in memory; plain pip is the fallback when uv cannot be had or the
    held versions cannot be satisfied, and the check below the fallback.
    """
    global _stale_after_install
    quiet = {"capture_output": True, "text": True}
    before = _loaded_distributions()
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as held:
        held.write("".join(f"{d}=={v}\n" for d, v in sorted(before.items())))
    run = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-qqq", "--disable-pip-version-check", "uv"], **quiet
    )
    if not run.returncode:
        run = subprocess.run(
            [sys.executable, "-m", "uv", "pip", "install", "--quiet", "--python", sys.executable,
             "--override", held.name, *packages],
            **quiet,
        )
    if run.returncode:
        run = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-qqq", "--disable-pip-version-check", *packages],
            **quiet,
        )
        if run.returncode:
            raise SystemExit(run.stdout[-3000:] + run.stderr[-3000:])
    # The safety net: anything loaded that changed on disk all the same needs a
    # restart. Kept in this session's globals: a re-run without the restart finds
    # nothing left to install, but the old copy is still the one in memory.
    after = _loaded_distributions()
    _stale_after_install = globals().get("_stale_after_install", {}) | {
        dist: f"{old} → {after[dist]}"
        for dist, old in before.items()
        if after.get(dist, old) != old
    }
    if _stale_after_install:
        raise SystemExit(
            "The install replaced packages this session had already loaded: "
            + ", ".join(f"{d} ({v})" for d, v in sorted(_stale_after_install.items()))
            + ".\nRuntime → Restart session, then run this cell again "
            "(or Run all). The second run is quick: nothing is installed again."
        )
# --- end of install helper ---


# Colab already has torch, transformers, h5py, pandas and tqdm; this only adds
# pyfaidx, and the `esm` SDK for an EvolutionaryScale model. That SDK pins
# transformers < 5, so the install replaces Colab's transformers 5 with a 4.x,
# which is why it is not installed for the other models.
_packages = ["h5py", "pandas", "pyfaidx", "torch", "tqdm", "transformers", "huggingface_hub"]
if globals().get("needs_esm_sdk"):
    _packages.append("esm")
print(f"Installing {', '.join(_packages)} (only what this runtime lacks)...")
_install(_packages)
print("Done.")

In [ ]:
# @title 3. Import libraries
import logging
import os
import re
from pathlib import Path

# transformers 4.x (what the `esm` SDK installs) imports TensorFlow and JAX
# whenever they are installed, as on Colab, to support their image types: that
# costs seconds and fills the output with TensorFlow's start-up log. This
# notebook runs on PyTorch only. transformers 5 dropped both; then these do
# nothing. Set before transformers is first imported.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("USE_JAX", "0")
os.environ.setdefault("USE_TORCH", "1")
# transformers 4.x also imports torchao, which Colab ships, and on a CPU runtime
# without Triton it logs "Detected no triton". Nothing here quantizes, so the
# warning has no consequence; there is no switch to keep transformers 4.x from
# importing it, so its logger is quieted instead.
logging.getLogger("torchao.kernel.intmm").setLevel(logging.ERROR)

try:
    import h5py
    import numpy as np
    import pandas as pd
    import torch
    from google.colab import drive, files, userdata
    from huggingface_hub import get_token
    from huggingface_hub import login as hf_login
    from pyfaidx import Fasta
    from tqdm.auto import tqdm
    from transformers import AutoTokenizer, EsmModel, T5EncoderModel, T5Tokenizer
except Exception as _exc:
    raise SystemExit(
        f"Import failed ({type(_exc).__name__}: {_exc}).\n"
        "Runtime → Restart session, then Run all again. Still failing after a "
        "restart? Report it at https://github.com/tsenoner/protspace/issues"
    ) from _exc

if torch.cuda.is_available():
    print(f"Ready: PyTorch {torch.__version__} on GPU ({torch.cuda.get_device_name(0)}).")
else:
    print(f"Ready: PyTorch {torch.__version__} on CPU. This is slow: Runtime → "
          "Change runtime type → T4 GPU, then Run all again.")

In [ ]:
# @title Optional: Hugging Face login (needed for the EvolutionaryScale models)
# @markdown ESM3 and ESM C are gated: accept their licence on huggingface.co while signed in,
# @markdown then paste a token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
# @markdown here, or store it as the Colab secret `HF_TOKEN` (key icon, left sidebar).
# @markdown The other models need no login.
hf_token_input = ""  # @param {type:"string"}

try:
    hf_token_secret = userdata.get("HF_TOKEN")
except Exception:
    hf_token_secret = None

if hf_token_input:
    hf_login(token=hf_token_input)
    print("Logged in with the token above.")
elif hf_token_secret:
    hf_login(token=hf_token_secret)
    print("Logged in with the Colab secret 'HF_TOKEN'.")
elif get_token() is not None:
    print("Already logged in to Hugging Face.")
elif needs_esm_sdk:
    print("Not logged in: loading this EvolutionaryScale model will fail. Paste a token "
          "above, or add the Colab secret 'HF_TOKEN', and run this cell again.")
else:
    print("Not logged in, and this model does not need it.")

In [ ]:
# @title Optional: Mount Google Drive
# @markdown Tick to read a FASTA from (or save the `.h5` to) your Google Drive; it asks
# @markdown for authorization the first time.
should_mount_drive = False  # @param {type:"boolean"}
if should_mount_drive:
    try:
        drive.mount("/content/drive")
        print("Google Drive mounted at /content/drive.")
    except Exception as e:
        print(f"Could not mount Google Drive: {e}")
else:
    print("Skipping Google Drive (tick the box to use it).")

In [ ]:
# @title 4. Choose the FASTA file
# @markdown Uses the path from step 1; with none, it asks you to upload a FASTA here.
if not fasta_filename:
    print("No path in step 1: pick a FASTA file to upload (wait for 'Saving ...').")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file received. Run this cell again and pick a FASTA file, "
                         "or enter its path in step 1.")
    fasta_filename = list(uploaded.keys())[0]

fasta_path = Path(fasta_filename).expanduser()
if not fasta_path.is_file():
    raise SystemExit(f"No file at {fasta_path}. Check the path in step 1 (for Google "
                     "Drive, tick 'Mount Google Drive' and use /content/drive/MyDrive/...).")
print(f"Using {fasta_path} ({fasta_path.stat().st_size / 1e6:.1f} MB).")

In [ ]:
# @title ### Functions { display-mode: "form" }

MODEL_SHORT_KEYS = {
    "Rostlab/prot_t5_xl_half_uniref50-enc": "prot_t5",
    "Rostlab/ProstT5_fp16": "prost_t5",
    "ElnaggarLab/ankh-base": "ankh_base",
    "ElnaggarLab/ankh-large": "ankh_large",
    "ElnaggarLab/ankh3-large": "ankh3_large",
    "facebook/esm2_t6_8M_UR50D": "esm2_8m",
    "facebook/esm2_t12_35M_UR50D": "esm2_35m",
    "facebook/esm2_t30_150M_UR50D": "esm2_150m",
    "facebook/esm2_t33_650M_UR50D": "esm2_650m",
    "EvolutionaryScale/esm3-sm-open-v1": "esm3_open",
    "EvolutionaryScale/esmc-300m-2024-12": "esmc_300m",
    "EvolutionaryScale/esmc-600m-2024-12": "esmc_600m",
}


def preprocess_sequences(df, model_type):
    """Prepare sequences for the given model type. Returns a new DataFrame."""
    df = df.copy()
    df["raw_sequence"] = df["sequence"]
    df["sequence"] = df["sequence"].str.replace("[BJOUZ]", "X", regex=True)
    if model_type in ("prost_t5", "prot_t5"):
        df["sequence"] = df["sequence"].apply(lambda s: " ".join(s))
    if model_type == "prost_t5":
        df["sequence"] = "<AA2fold> " + df["sequence"]
    return df


def _load_failure(checkpoint, exc):
    """SystemExit saying why a checkpoint did not load and what to do."""
    text = f"{type(exc).__name__}: {exc}"
    if any(s in text for s in ("401", "403", "GatedRepo", "gated", "Unauthorized")):
        why = ("it is gated: accept its licence on huggingface.co while signed in, then "
               "log in with the 'Optional: Hugging Face login' step and run this one again.")
    elif any(s in text for s in ("ConnectionError", "Timeout", "timed out", "Max retries")):
        why = "Hugging Face did not answer: check the connection and run this step again."
    elif isinstance(exc, ImportError):
        why = ("a package it needs is missing: run step 1 and step 2 again (step 2 "
               "installs the `esm` SDK only for the EvolutionaryScale models).")
    else:
        why = "see the error below."
    return SystemExit(f"Could not load {checkpoint}: {why}\n{text[:1500]}")


def setup_model(checkpoint):
    """Load model and tokenizer for the given checkpoint."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading {checkpoint} on {device} (the first time downloads its weights)...")
    # The checkpoints carry heads an encoder-only load leaves out (ESM2's masked-LM
    # head, Ankh's decoder), and transformers 5 prints a table of those keys as
    # UNEXPECTED on every load. That is the intended outcome here, so its loading
    # messages are kept to errors while the weights load.
    from transformers.utils import logging as hf_logging

    verbosity = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()
    try:
        return _setup_model(checkpoint, device)
    except (SystemExit, KeyboardInterrupt):
        raise
    except Exception as exc:
        raise _load_failure(checkpoint, exc) from exc
    finally:
        hf_logging.set_verbosity(verbosity)


def _setup_model(checkpoint, device):

    if "esm2" in checkpoint:
        mod_type = "esm_transformer"
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        # No pooler: only the per-residue states are used, and the checkpoints
        # ship none, so it would be randomly initialised and reported MISSING.
        model = EsmModel.from_pretrained(checkpoint, add_pooling_layer=False)
    elif "esm3-sm-open-v1" in checkpoint:
        from esm.models.esm3 import ESM3  # the `esm` SDK, installed only for these

        mod_type = "native_esm3"
        tokenizer = None
        model = ESM3.from_pretrained("esm3-open")
    elif checkpoint == "EvolutionaryScale/esmc-300m-2024-12":
        from esm.models.esmc import ESMC

        mod_type = "native_esmc"
        tokenizer = None
        model = ESMC.from_pretrained("esmc_300m")
    elif checkpoint == "EvolutionaryScale/esmc-600m-2024-12":
        from esm.models.esmc import ESMC

        mod_type = "native_esmc"
        tokenizer = None
        model = ESMC.from_pretrained("esmc_600m")
    elif "ankh" in checkpoint:
        mod_type = "ankh"
        tokenizer = AutoTokenizer.from_pretrained(checkpoint)
        model = T5EncoderModel.from_pretrained(checkpoint)
    elif "prot_t5" in checkpoint or "ProstT5" in checkpoint:
        is_prostt5 = "prostt5" in checkpoint.lower()
        mod_type = "prost_t5" if is_prostt5 else "prot_t5"
        tokenizer = T5Tokenizer.from_pretrained(
            checkpoint, do_lower_case=not is_prostt5
        )
        model = T5EncoderModel.from_pretrained(checkpoint, torch_dtype=torch.float16)
        if device.type == "cuda":
            model = model.half()
    else:
        raise ValueError(f"Unknown model checkpoint: {checkpoint}")

    return model.to(device), tokenizer, mod_type


def read_fasta(file_path):
    """Read a FASTA file and return headers and sequences."""
    headers, sequences = [], []
    with Fasta(str(file_path)) as fasta_data:
        for record in fasta_data:
            headers.append(record.name)
            sequences.append(str(record))
    return headers, sequences


def _embed_native_esm(sequence, emb_type, mod_type, model, device):
    """Compute embeddings using native ESM3/ESMC models."""
    from esm.sdk.api import ESMProtein, LogitsConfig, SamplingConfig

    protein = ESMProtein(sequence=sequence)
    tokens = model.encode(protein).to(device)

    with torch.inference_mode():
        if mod_type == "native_esm3":
            out = model.forward_and_sample(
                tokens, SamplingConfig(return_per_residue_embeddings=True)
            )
            per_res = out.per_residue_embedding.squeeze(0).cpu().float().numpy()
        else:  # native_esmc
            out = model.logits(
                tokens, LogitsConfig(sequence=True, return_embeddings=True)
            )
            per_res = out.embeddings.squeeze(0).cpu().float().numpy()

    per_res = per_res[1:-1, :]  # strip BOS/EOS
    return per_res.mean(axis=0) if emb_type == "per_prot" else per_res


def _embed_transformers_batch(sequences, emb_type, mod_type, model, tokenizer, max_len):
    """Compute embeddings for a batch using HuggingFace Transformers."""
    inputs = tokenizer(
        sequences,
        return_tensors="pt",
        max_length=max_len + 2,
        truncation=True,
        padding=True,
        add_special_tokens=True,
    ).to(model.device)

    with torch.inference_mode():
        hidden = model(**inputs).last_hidden_state.cpu().float().numpy()

    lengths = inputs["attention_mask"].sum(dim=1).tolist()

    results = []
    for i, seq_len in enumerate(lengths):
        # Strip special tokens: ESM2/ProstT5 have <cls>/<AA2fold> and <eos>,
        # ProtT5/Ankh have only <eos>
        if mod_type in ("esm_transformer", "prost_t5"):
            emb = hidden[i, 1 : seq_len - 1, :]
        else:  # prot_t5, ankh
            emb = hidden[i, : seq_len - 1, :]
        results.append(emb.mean(axis=0) if emb_type == "per_prot" else emb)

    return results


def _write_batch(hdf, headers, embeddings):
    """Write a batch of embeddings to an open HDF5 file."""
    for header, emb in zip(headers, embeddings, strict=False):
        hdf.create_dataset(name=header, data=emb.astype(np.float32))


def create_embedding(
    checkpoint,
    df,
    emb_type="per_prot",
    output_file="protein_embeddings.h5",
    max_len=2000,
    batch_size=32,
    model_key=None,
):
    """Generate embeddings and write them to an HDF5 file.

    ``model_key`` is stored as the file's ``model_name`` attribute, which is how
    ProtSpace names the embedding (and labels its projections) when it loads it.
    """
    print("Setting up model...")
    model, tokenizer, mod_type = setup_model(checkpoint)
    model.eval()
    df_proc = preprocess_sequences(df, mod_type)

    # Validate headers (HDF5 doesn't allow '/' in dataset names)
    bad = df_proc[df_proc["header"].str.contains("/")]["header"].tolist()
    if bad:
        raise SystemExit(
            f"{len(bad)} FASTA header(s) contain '/', which an HDF5 name cannot hold, e.g.:\n"
            + "\n".join(bad[:10])
            + "\nReplace the '/' in those headers, or tick 'Use the UniProt accession' in "
            "step 1 if they are UniProt headers, and run step 5 again."
        )

    with h5py.File(output_file, "a") as hdf:
        if model_key:
            hdf.attrs["model_name"] = model_key
        # Skip already-computed embeddings via set comparison
        existing = set(hdf.keys())
        already_done = df_proc["header"].isin(existing)
        too_long = df_proc["raw_sequence"].str.len() > max_len
        to_compute = df_proc[~already_done & ~too_long].copy()

        n_existing = already_done.sum()
        n_too_long = (~already_done & too_long).sum()
        if n_existing:
            print(f"Skipping {n_existing} already computed embeddings")
        if n_too_long:
            longest = ", ".join(df_proc.loc[~already_done & too_long, "header"].head(5))
            print(f"Skipping {n_too_long} sequence(s) longer than {max_len} residues "
                  f"(e.g. {longest}); raise the maximum length in step 1 to include them.")

        if to_compute.empty:
            print("All embeddings already computed!")
            return

        # Sort by sequence length for efficient batching (shortest first)
        to_compute = to_compute.sort_values(
            by="raw_sequence", key=lambda s: s.str.len()
        )

        is_native = mod_type.startswith("native_esm")
        device = model.device
        print(f"Computing {len(to_compute)} embeddings, shortest first...")

        if is_native:
            # Native ESM: single sequence at a time (API limitation)
            for _, row in tqdm(to_compute.iterrows(), total=len(to_compute), desc="Embedding", unit="seq"):
                emb = _embed_native_esm(
                    row["sequence"], emb_type, mod_type, model, device
                )
                hdf.create_dataset(name=row["header"], data=emb.astype(np.float32))
        else:
            # Transformers: batched with adaptive OOM recovery.
            # Sequences are sorted shortest-first, so once we reduce the
            # batch size we never need to increase it again.
            pbar = tqdm(total=len(to_compute), desc="Embedding", unit="seq")
            i = 0
            bs = batch_size
            while i < len(to_compute):
                batch = to_compute.iloc[i : i + bs]
                try:
                    embeddings = _embed_transformers_batch(
                        batch["sequence"].tolist(),
                        emb_type,
                        mod_type,
                        model,
                        tokenizer,
                        max_len,
                    )
                    _write_batch(hdf, batch["header"], embeddings)
                    pbar.update(len(batch))
                    i += bs
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if bs > 1:
                        bs = max(1, bs // 2)
                        tqdm.write(f"GPU OOM — reducing batch size to {bs}")
                    else:
                        tqdm.write(
                            f"Skipping {batch.iloc[0]['header']} "
                            f"(len={len(batch.iloc[0]['raw_sequence'])}, OOM at batch_size=1)"
                        )
                        i += 1
                        pbar.update(1)
            pbar.close()

    del model, tokenizer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# @title 5. Generate embeddings
# @markdown Writes `<fasta name>_<model>.h5` next to the FASTA. A re-run resumes: proteins
# @markdown already in that file are skipped.

# Get the short key for the selected model_name, fallback to sanitized full name
short_model_key = MODEL_SHORT_KEYS.get(model_name, model_name.replace("/", "_"))
output_file = str(fasta_path.with_name(f"{fasta_path.stem}_{short_model_key}.h5"))

try:
    headers, sequences = read_fasta(fasta_path)
except Exception as exc:
    raise SystemExit(f"Could not read {fasta_path} as FASTA ({type(exc).__name__}: {exc}). "
                     "Check that it is a FASTA file: '>' header lines, each followed by "
                     "its sequence.") from exc
if not headers:
    raise SystemExit(f"No FASTA records ('>' lines) in {fasta_path}.")
df = pd.DataFrame({"header": headers, "sequence": sequences})
print(f"Read {len(df):,} sequences from {fasta_path.name}.")

# Regex to extract UniProt ID
uniprot_regex = r"[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z][A-Z0-9]{2}[0-9]){1,2}"

if should_extract_uniprot_id:
    df["header"] = df["header"].apply(
        lambda header: (
            re.search(uniprot_regex, header).group(0)
            if re.search(uniprot_regex, header)
            else header
        )
    )
    print("Using the UniProt accession in each header as its ID (full header where none).")
else:
    print("Using the full FASTA headers as IDs.")
duplicated = df["header"][df["header"].duplicated()].unique()
if len(duplicated):
    raise SystemExit(f"{len(duplicated)} ID(s) appear more than once, e.g. "
                     f"{', '.join(duplicated[:5])}: an .h5 holds one vector per ID. "
                     "Remove the duplicates from the FASTA and run this step again.")

create_embedding(
    model_name,
    df,
    emb_type=embedding_type,
    output_file=output_file,
    max_len=max_sequence_length,
    batch_size=batch_size,
    model_key=short_model_key,
)

with h5py.File(output_file, "r") as _h5:
    _written = len(_h5)
_missing = len(df) - _written
print(f"\nSaved {_written:,} embeddings to {output_file} "
      f"({Path(output_file).stat().st_size / 1e6:.1f} MB), named '{short_model_key}'.")
if _missing:
    print(f"{_missing:,} sequence(s) have no embedding (too long or out of memory; listed above).")
print("Next: step 6 downloads it; load it in the Preparation notebook's Embeddings (.h5) tab.")

In [ ]:
# @title 6. Download the embeddings
# @markdown Downloads the `.h5` to your computer (it stays in this runtime too).
if "output_file" not in globals() or not Path(output_file).is_file():
    raise SystemExit("No embeddings file yet: run step 5 first.")
print(f"Downloading {Path(output_file).name}; if the browser blocks it, take it from the "
      "Files pane (left sidebar).")
files.download(output_file)